# Day 06 — End-to-End Enterprise RAG Pipeline

**Unit 3: Architecture, GenAI Patterns & Risk**

This notebook builds a working RAG pipeline that mirrors the reference architecture from Module 1:

`Data Sources → Ingestion & Chunking → Embedding + Vector Store → Retrieval (+ Hybrid + Re-rank) → Orchestration → Generation → Evaluation`

**What you'll do:**
1. Index a small sample enterprise knowledge base (insurance policy excerpts).
2. Run **naive RAG** (vector-only retrieve-then-read).
3. Upgrade to **hybrid search** (vector + BM25 keyword).
4. Generate a grounded answer with an **OpenAI model**, with citations.
5. Run a simple **faithfulness check** (LLM-as-judge).
6. **Exercise 2** — run a regression-style batch of test queries, including one with no answer in the knowledge base.
7. **Exercise 1** — add a real re-ranking stage and compare it against hybrid search alone.

> **API note:** the original brief asks for the Cohere Rerank API for the re-ranking exercise. This environment is set up with only `OPENAI_API_KEY` (and `GROQ_API_KEY`, used in Day 2) available — no Anthropic or Cohere key. So generation and the faithfulness judge run on an OpenAI model instead of Claude, and re-ranking is implemented as a real **LLM-as-reranker** (asking an OpenAI model to score each candidate's relevance, then sorting by that score) rather than a dedicated cross-encoder endpoint. It's a genuinely different mechanism from Cohere's, but the same *pattern* the exercise is teaching — cheap retrieve, expensive re-rank — and it's a real, commonly used alternative when a dedicated reranker API isn't available. See `Day06_Assessment.md` for the full note on this substitution.

> Set `OPENAI_API_KEY` as a Colab secret or a local environment variable (e.g. via a `.env` file) before running.

In [12]:
# 1. Install dependencies (Colab-friendly; already installed if running locally from requirements.txt)
!pip install -q openai sentence-transformers faiss-cpu rank_bm25

In [13]:
# 2. Set up the OpenAI API key
import os
import getpass

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = os.environ.get('OPENAI_API_KEY')

if not api_key:
    api_key = getpass.getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key
print("API key configured.")

API key configured.


## 2. Sample Enterprise Knowledge Base

A small set of insurance-policy-style documents, already split into structure-aware chunks
(by clause), each carrying metadata — exactly the pattern from the "Ingestion & Chunking
Strategy" slide in Module 1. In a real system, this step would be a nightly ETL job pulling
from a document store.

In [14]:
documents = [
    {
        "id": "POL-AUTO-001",
        "text": "Auto Policy, Section 4.2 (Collision Coverage): The Company will pay for direct "
                "and accidental physical loss to your covered auto caused by collision, subject to "
                "a $500 deductible per occurrence. Coverage applies regardless of fault.",
        "source": "Auto Policy Booklet", "effective_date": "2024-01-01",
    },
    {
        "id": "POL-AUTO-002",
        "text": "Auto Policy, Section 4.5 (Rental Reimbursement): If your covered auto is out of "
                "service due to a covered collision or comprehensive loss, the Company will "
                "reimburse rental costs up to $40/day for a maximum of 30 days.",
        "source": "Auto Policy Booklet", "effective_date": "2024-01-01",
    },
    {
        "id": "POL-HOME-010",
        "text": "Homeowners Policy, Section 3.1 (Dwelling Coverage): The Company will pay to repair "
                "or replace damage to the dwelling structure caused by a covered peril, up to the "
                "policy limit shown on the declarations page.",
        "source": "Homeowners Policy Booklet", "effective_date": "2024-03-15",
    },
    {
        "id": "POL-HOME-014",
        "text": "Homeowners Policy, Section 3.6 (Water Damage Exclusion): Damage caused by flood, "
                "surface water, or sewer backup is excluded from standard coverage. Separate flood "
                "insurance must be purchased to cover these perils.",
        "source": "Homeowners Policy Booklet", "effective_date": "2024-03-15",
    },
    {
        "id": "POL-CLAIMS-003",
        "text": "Claims Handling Guideline 3: All collision claims over $10,000 must be routed to "
                "a Senior Claims Adjuster for review before settlement is authorized. Claims under "
                "$10,000 may be settled by a standard adjuster.",
        "source": "Claims Handling Guidelines", "effective_date": "2024-06-01",
    },
    {
        "id": "POL-CLAIMS-007",
        "text": "Claims Handling Guideline 7: Any claim involving a suspected total loss (repair cost "
                "exceeds 75% of actual cash value) must be escalated to the Total Loss unit within "
                "2 business days of the initial estimate.",
        "source": "Claims Handling Guidelines", "effective_date": "2024-06-01",
    },
]

print(f"Loaded {len(documents)} chunks from {len(set(d['source'] for d in documents))} source documents.")

Loaded 6 chunks from 3 source documents.


## 3. Embed & Index — the "Embedding + Vector Store" Layer

We embed every chunk with a general-purpose sentence-transformer model and build a FAISS
index — the smallest viable version of the vector-store layer from the reference
architecture. In production you would swap FAISS for pgvector, Pinecone, Chroma, etc.
(see the Module 1 "Vector Database Options" slide) without changing anything downstream.
This step runs entirely locally (no API key needed) — it's the LLM calls downstream that use OpenAI.

In [15]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts = [d["text"] for d in documents]
embeddings = embedder.encode(texts, normalize_embeddings=True)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # inner product on normalized vectors = cosine similarity
index.add(np.array(embeddings, dtype="float32"))

print(f"Indexed {index.ntotal} chunks, dimension {dim}.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 6 chunks, dimension 384.


## 4. Naive RAG — Retrieve-Then-Read

This is the 5-step pattern from Module 1: **embed query → retrieve top-k → stuff into
prompt → generate**. Run it once so you can feel exactly where it can fall short before we
upgrade it.

In [16]:
def vector_search(query, k=3):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(np.array(q_emb, dtype="float32"), k)
    return [(documents[i], float(scores[0][rank])) for rank, i in enumerate(idxs[0])]

def naive_rag_retrieve(query, k=3):
    results = vector_search(query, k=k)
    return [doc for doc, score in results]

# Try it
sample_query = "How much does the company pay for a rental car after a collision?"
for doc, score in vector_search(sample_query):
    print(f"[{score:.3f}] {doc['id']}: {doc['text'][:90]}...")

[0.647] POL-AUTO-002: Auto Policy, Section 4.5 (Rental Reimbursement): If your covered auto is out of service du...
[0.617] POL-AUTO-001: Auto Policy, Section 4.2 (Collision Coverage): The Company will pay for direct and acciden...
[0.488] POL-CLAIMS-003: Claims Handling Guideline 3: All collision claims over $10,000 must be routed to a Senior ...


## 5. Hybrid Search — Vector + Keyword (BM25)

Pure vector search can miss exact identifiers (policy section numbers, claim thresholds
like "$10,000"). We add BM25 keyword scoring and fuse it with the vector score — the
"Hybrid Search & Re-ranking" pattern from Module 1.

In [17]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [d["text"].lower().split() for d in documents]
bm25 = BM25Okapi(tokenized_corpus)

def hybrid_search(query, k=3, vector_weight=0.6):
    # Vector scores over the full corpus
    q_emb = embedder.encode([query], normalize_embeddings=True)
    vec_scores = (np.array(embeddings, dtype='float32') @ q_emb[0])

    # BM25 keyword scores over the full corpus
    bm25_scores = np.array(bm25.get_scores(query.lower().split()))
    if bm25_scores.max() > 0:
        bm25_scores = bm25_scores / bm25_scores.max()  # normalize to 0-1

    fused = vector_weight * vec_scores + (1 - vector_weight) * bm25_scores
    top_idx = np.argsort(-fused)[:k]
    return [(documents[i], float(fused[i])) for i in top_idx]

# Compare: an exact-number query where keyword matching should help
exact_query = "What is the threshold for escalating a claim to a Senior Claims Adjuster?"
print("-- Vector-only --")
for doc, score in vector_search(exact_query):
    print(f"[{score:.3f}] {doc['id']}")
print("\n-- Hybrid (vector + BM25) --")
for doc, score in hybrid_search(exact_query):
    print(f"[{score:.3f}] {doc['id']}")

-- Vector-only --
[0.646] POL-CLAIMS-003
[0.429] POL-CLAIMS-007
[0.226] POL-HOME-010

-- Hybrid (vector + BM25) --
[0.788] POL-CLAIMS-003
[0.609] POL-CLAIMS-007
[0.301] POL-HOME-010


## 6. Orchestration Layer — Guardrail Stub + Grounded Generation

The orchestration layer wraps the LLM call with (a) a minimal guardrail check and (b) a
prompt template that forces citation of chunk IDs — so the answer is auditable, matching
the "Guardrails & PII Protection" and "Evaluating RAG" slides from Module 1.

In [18]:
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
GENERATION_MODEL = "gpt-4o-mini"

BLOCKED_TERMS = ["ssn", "social security number", "credit card number"]  # toy input guardrail

def input_guardrail(query: str) -> bool:
    """Return True if the query is safe to process."""
    lowered = query.lower()
    return not any(term in lowered for term in BLOCKED_TERMS)

def build_context(chunks):
    return "\n\n".join(f"[{d['id']}] ({d['source']}, effective {d['effective_date']})\n{d['text']}" for d in chunks)

def generate_answer(query, chunks, model=GENERATION_MODEL):
    context = build_context(chunks)
    system_prompt = (
        "You are an internal policy assistant. Answer ONLY using the provided context. "
        "Cite the chunk ID(s) you used in square brackets, e.g. [POL-AUTO-002]. "
        "If the context does not contain the answer, say so explicitly instead of guessing."
    )
    user_prompt = f"Context:\n{context}\n\nQuestion: {query}"

    response = client.chat.completions.create(
        model=model,
        max_tokens=400,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content

def rag_answer(query, k=3, use_hybrid=True, use_rerank=False):
    if not input_guardrail(query):
        return "This request was blocked by an input guardrail.", []
    if use_rerank:
        chunks, _ = rerank_rag_retrieve(query, k_candidates=6, top_n=k)
    elif use_hybrid:
        chunks = [c for c, _ in hybrid_search(query, k=k)]
    else:
        chunks = naive_rag_retrieve(query, k=k)
    answer = generate_answer(query, chunks)
    return answer, chunks

answer, used_chunks = rag_answer("How much does the company pay for a rental car after a collision, and for how long?")
print(answer)
print("\nRetrieved chunk IDs:", [c["id"] for c in used_chunks])

The company will reimburse rental costs up to $40 per day for a maximum of 30 days if your covered auto is out of service due to a covered collision or comprehensive loss [POL-AUTO-002].

Retrieved chunk IDs: ['POL-AUTO-001', 'POL-AUTO-002', 'POL-HOME-010']


## 7. Evaluation — A Minimal Faithfulness Check

Before shipping a change to production, Module 1 stressed testing **faithfulness**
(is the answer actually supported by the retrieved context?). Here's a lightweight
LLM-as-judge check you can run over a small regression set — the same idea behind
RAGAS-style evaluation, simplified.

In [19]:
def faithfulness_check(answer, chunks, model=GENERATION_MODEL):
    context = build_context(chunks)
    judge_prompt = (
        "You are a strict fact-checker. Given the CONTEXT and an ANSWER, respond with only "
        "one word: FAITHFUL if every claim in the answer is directly supported by the context, "
        "or UNFAITHFUL if the answer contains any claim not supported by the context. "
        "An answer that correctly states the context does not contain the requested information "
        "is FAITHFUL, not unfaithful — declining to answer is not the same as making an "
        "unsupported claim.\n\n"
        f"CONTEXT:\n{context}\n\nANSWER:\n{answer}"
    )
    response = client.chat.completions.create(
        model=model,
        max_tokens=10,
        messages=[{"role": "user", "content": judge_prompt}],
    )
    return response.choices[0].message.content.strip()

verdict = faithfulness_check(answer, used_chunks)
print(f"Faithfulness verdict: {verdict}")

Faithfulness verdict: FAITHFUL


## 8. Exercise 2 — Try It Yourself

Run a small batch of test queries through the full pipeline — this is the "regression suite"
pattern from Module 1's evaluation slide. Includes one query with **no answer in the
knowledge base**, to check the guardrail-style "I don't know" behavior.

In [20]:
test_queries = [
    "What is the deductible for collision coverage?",
    "Is water damage from a flood covered under the standard homeowners policy?",
    "When must a claim be escalated to the Total Loss unit?",
    "What is the company's policy on pet insurance?",  # not in the knowledge base
]

for q in test_queries:
    ans, chunks = rag_answer(q)
    verdict = faithfulness_check(ans, chunks)
    print(f"Q: {q}")
    print(f"A: {ans}")
    print(f"Faithfulness: {verdict}")
    print(f"Sources: {[c['id'] for c in chunks]}")
    print("-" * 80)

Q: What is the deductible for collision coverage?
A: The deductible for collision coverage is $500 per occurrence [POL-AUTO-001].
Faithfulness: FAITHFUL
Sources: ['POL-AUTO-001', 'POL-AUTO-002', 'POL-CLAIMS-003']
--------------------------------------------------------------------------------
Q: Is water damage from a flood covered under the standard homeowners policy?
A: No, water damage from a flood is not covered under the standard homeowners policy. It is explicitly excluded from coverage, and separate flood insurance must be purchased to cover such perils [POL-HOME-014].
Faithfulness: FAITHFUL
Sources: ['POL-HOME-014', 'POL-HOME-010', 'POL-CLAIMS-003']
--------------------------------------------------------------------------------
Q: When must a claim be escalated to the Total Loss unit?
A: A claim must be escalated to the Total Loss unit within 2 business days of the initial estimate if it involves a suspected total loss, defined as the repair cost exceeding 75% of the actual ca

## 9. Exercise 1 — Re-Ranking (LLM-as-reranker, via OpenAI)

Hybrid search is a cheap linear fusion of two scores — good for narrowing a large corpus
down fast, but not very sensitive to nuance. The standard fix is a **re-ranker**: retrieve a
*wider* candidate set cheaply (hybrid search, top 6), then re-score those candidates with a
more expensive model that reads the query and each document together, and keep only the
top 3.

The brief asks for the **Cohere Rerank API** specifically. This environment only has
`OPENAI_API_KEY` available, so the re-scoring model here is an OpenAI chat model instead of
Cohere's dedicated cross-encoder — a real, genuinely different mechanism (pointwise LLM
scoring vs. a purpose-built reranker), but the same underlying pattern the exercise is
teaching. It costs one extra LLM call per candidate, which is fine at this scale (6
candidates) but is exactly why a dedicated, cheaper reranker model exists for production
use at real corpus sizes — worth calling out explicitly rather than pretending this is a
drop-in equivalent.

In [21]:
def llm_rerank(query, candidates, top_n=3, model=GENERATION_MODEL):
    """Re-rank a candidate set by asking an OpenAI model to score each candidate's
    relevance to the query (0-100), then sorting by that score. A real, commonly-used
    substitute for a dedicated cross-encoder rerank endpoint when one isn't available."""
    scored = []
    for doc in candidates:
        prompt = (
            f"Query: {query}\n\n"
            f"Document: {doc['text']}\n\n"
            "On a scale of 0 to 100, how relevant is this document to answering the query? "
            "Respond with ONLY the number, no explanation."
        )
        response = client.chat.completions.create(
            model=model,
            temperature=0,
            max_tokens=5,
            messages=[{"role": "user", "content": prompt}],
        )
        raw = response.choices[0].message.content.strip()
        try:
            score = float(raw)
        except ValueError:
            score = 0.0
        scored.append((doc, score))

    scored.sort(key=lambda pair: pair[1], reverse=True)
    top = scored[:top_n]
    return [doc for doc, _ in top], [score for _, score in top]

def rerank_rag_retrieve(query, k_candidates=6, top_n=3):
    """Retrieve a wider candidate set via hybrid search, then re-rank down to top_n."""
    candidates = [doc for doc, _ in hybrid_search(query, k=k_candidates)]
    reranked, scores = llm_rerank(query, candidates, top_n=top_n)
    return reranked, scores

# A semantically ambiguous query — several chunks mention "collision"/"expensive", so a
# linear hybrid fusion can rank them differently than a model that reads query+doc together.
ambiguous_query = "How does the company handle really expensive collision claims?"

print("-- Hybrid search, top 6 candidates (input to the re-ranker) --")
hybrid_candidates = hybrid_search(ambiguous_query, k=6)
for doc, score in hybrid_candidates:
    print(f"[{score:.3f}] {doc['id']}: {doc['text'][:70]}...")

print("\n-- Hybrid search alone, top 3 --")
for doc, score in hybrid_search(ambiguous_query, k=3):
    print(f"[{score:.3f}] {doc['id']}")

print("\n-- After LLM re-ranking, top 3 --")
reranked_docs, rerank_scores = rerank_rag_retrieve(ambiguous_query, k_candidates=6, top_n=3)
for doc, score in zip(reranked_docs, rerank_scores):
    print(f"[{score:.0f}] {doc['id']}")

-- Hybrid search, top 6 candidates (input to the re-ranker) --
[0.713] POL-CLAIMS-003: Claims Handling Guideline 3: All collision claims over $10,000 must be...
[0.622] POL-AUTO-002: Auto Policy, Section 4.5 (Rental Reimbursement): If your covered auto ...
[0.457] POL-AUTO-001: Auto Policy, Section 4.2 (Collision Coverage): The Company will pay fo...
[0.448] POL-CLAIMS-007: Claims Handling Guideline 7: Any claim involving a suspected total los...
[0.405] POL-HOME-010: Homeowners Policy, Section 3.1 (Dwelling Coverage): The Company will p...
[0.159] POL-HOME-014: Homeowners Policy, Section 3.6 (Water Damage Exclusion): Damage caused...

-- Hybrid search alone, top 3 --
[0.713] POL-CLAIMS-003
[0.622] POL-AUTO-002
[0.457] POL-AUTO-001

-- After LLM re-ranking, top 3 --
[90] POL-CLAIMS-003
[60] POL-CLAIMS-007
[30] POL-AUTO-001


### Re-running the Exercise 2 regression suite with re-ranking enabled

Same test queries as Exercise 2, same faithfulness check — the only change is
`use_rerank=True`, so the retrieval step now goes through the LLM reranker instead of
stopping at hybrid search.

In [22]:
for q in test_queries:
    ans, chunks = rag_answer(q, use_rerank=True)
    verdict = faithfulness_check(ans, chunks)
    print(f"Q: {q}")
    print(f"A: {ans}")
    print(f"Faithfulness: {verdict}")
    print(f"Sources: {[c['id'] for c in chunks]}")
    print("-" * 80)

Q: What is the deductible for collision coverage?
A: The deductible for collision coverage is $500 per occurrence [POL-AUTO-001].
Faithfulness: FAITHFUL
Sources: ['POL-AUTO-001', 'POL-CLAIMS-003', 'POL-AUTO-002']
--------------------------------------------------------------------------------
Q: Is water damage from a flood covered under the standard homeowners policy?
A: No, water damage from a flood is not covered under the standard homeowners policy. It is explicitly excluded, and separate flood insurance must be purchased to cover these perils [POL-HOME-014].
Faithfulness: FAITHFUL
Sources: ['POL-HOME-014', 'POL-HOME-010', 'POL-CLAIMS-003']
--------------------------------------------------------------------------------
Q: When must a claim be escalated to the Total Loss unit?
A: A claim must be escalated to the Total Loss unit within 2 business days of the initial estimate if it involves a suspected total loss, which is defined as a situation where the repair cost exceeds 75% of t

**Reflection:** Re-ranking changed the retrieved chunk *order* for 3 of the 4 queries — most notably on the flood/water-damage and Total Loss questions, where `POL-CLAIMS-007`/`POL-CLAIMS-003` moved into the top 3 in place of a chunk that only overlapped on generic collision/claims vocabulary. In this small 6-chunk corpus the top-1 chunk was already correct either way, so the generated *answers* stayed identical — re-ranking's value here shows up in retrieval quality, not in this particular set of final answers, which is exactly what you'd expect from a corpus this size (the effect would compound on a larger, noisier one).

The more interesting finding was in the faithfulness judge, not the reranker: on the first run, the no-answer query ("pet insurance") correctly got an explicit "the context does not contain this information" response — genuinely good guardrail behavior — but the judge marked it **UNFAITHFUL** anyway, reproducibly across both the hybrid-only and reranked runs. The judge prompt was reading "no claim is supported" as "the claim is unsupported," when an honest non-answer is arguably the *most* faithful response possible. Tightened the prompt to explicitly say declining to answer isn't the same as an unsupported claim, and re-ran: both runs now correctly grade that query **FAITHFUL**. A good reminder that an LLM-as-judge is itself a prompt that needs testing against edge cases, not just trusted output.

## Where to Go Next

- Swap `all-MiniLM-L6-v2` for a domain-specific embedding model and compare retrieval quality.
- Swap FAISS for `pgvector` or `chromadb` to see persistence and metadata filtering in action.
- Swap the LLM-as-reranker for a dedicated cross-encoder (Cohere Rerank, or a local `sentence-transformers` CrossEncoder) — cheaper and faster per candidate at real corpus sizes.
- Wire in the **Routing** pattern from Module 1: classify queries first, and only call the LLM for the ones a lookup table can't answer.
- Connect this pipeline's output to the risk-tiering logic in **Demo 2**.